In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn pandas numpy

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score, classification_report

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

In [ ]:
#Load dataset
FILE_PATH = "/content/Sentences_75Agree.txt"

with open(FILE_PATH, "r", encoding="utf-8", errors="replace") as f:
    lines = f.readlines()

rows = []
for line in lines:
    line = line.strip()
    if not line:
        continue

    parts = line.rsplit("@", 1)
    if len(parts) != 2:
        continue

    text, label = parts
    text = text.strip()
    label = label.strip().lower()

    if text and label in ["positive", "neutral", "negative"]:
        rows.append((text, label))

df = pd.DataFrame(rows, columns=["text", "label"])

print("Shape:", df.shape)
print(df["label"].value_counts())

In [ ]:
#Encoding labels
label_encoder = LabelEncoder()
df["label_id"] = label_encoder.fit_transform(df["label"])

X = df["text"].tolist()
y = df["label_id"].tolist()

In [ ]:
#Train/test split
X_pool, X_test, y_pool, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

pool_df = pd.DataFrame({
    "text": X_pool,
    "label": y_pool
})

print("Pool class counts:")
print(pool_df["label"].value_counts().sort_index())

In [ ]:
#Build balanced initial seed
INITIAL_PER_CLASS = 60   # total = 180
QUERY_SIZE = 30
N_ROUNDS = 5
RANDOM_STATE = 42

labeled_parts = []
remaining_parts = []

for cls in sorted(pool_df["label"].unique()):
    class_subset = pool_df[pool_df["label"] == cls]

    sampled = class_subset.sample(
        n=INITIAL_PER_CLASS,
        random_state=RANDOM_STATE
    )
    labeled_parts.append(sampled)

    remaining = class_subset.drop(sampled.index)
    remaining_parts.append(remaining)

initial_labeled_df = pd.concat(labeled_parts).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
initial_unlabeled_df = pd.concat(remaining_parts).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print("\nInitial labeled class counts:")
print(initial_labeled_df["label"].value_counts().sort_index())

print("\nInitial unlabeled pool class counts:")
print(initial_unlabeled_df["label"].value_counts().sort_index())

In [ ]:
#Tokenizer
MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
#Tokenize the datasets (helper)
def make_hf_dataset(df_input):
    ds = Dataset.from_pandas(df_input[["text", "label"]].copy())

    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=128
        )

    ds = ds.map(tokenize_function, batched=True)
    ds = ds.remove_columns(["text"])
    ds.set_format("torch")
    return ds

#Fixed test set
test_df = pd.DataFrame({
    "text": X_test,
    "label": y_test
})
test_dataset = make_hf_dataset(test_df)

In [ ]:
#Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

In [ ]:
#Train and evaluate RoBERTa (helper)
def train_and_evaluate_roberta(train_df, run_name="roberta_al_run"):
    train_dataset = make_hf_dataset(train_df)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(label_encoder.classes_)
    )

    training_args = TrainingArguments(
        output_dir=f"./{run_name}",
        eval_strategy="epoch",
        save_strategy="no",
        logging_strategy="no",
        num_train_epochs=5,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=64,
        learning_rate=2e-5,
        weight_decay=0.01,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    trainer.train()
    eval_results = trainer.evaluate()

    return trainer, eval_results

In [ ]:
#Score unlabeled pool (helper)
def get_unlabeled_probs(trainer, unlabeled_df):
    unlabeled_dataset = make_hf_dataset(unlabeled_df)
    pred_output = trainer.predict(unlabeled_dataset)
    logits = pred_output.predictions

    # softmax
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    return probs

In [ ]:
#Query strategies (random, entropy, margin)
def query_random(unlabeled_df, query_size, random_state=42):
    return unlabeled_df.sample(n=query_size, random_state=random_state).index

def query_entropy(unlabeled_df, probs, query_size):
    entropy = -np.sum(probs * np.log(probs + 1e-12), axis=1)
    top_idx = np.argsort(-entropy)[:query_size]
    return unlabeled_df.iloc[top_idx].index

def query_margin(unlabeled_df, probs, query_size):
    sorted_probs = np.sort(probs, axis=1)
    margins = sorted_probs[:, -1] - sorted_probs[:, -2]
    top_idx = np.argsort(margins)[:query_size]   # smallest margin = most uncertain
    return unlabeled_df.iloc[top_idx].index

In [ ]:
#Running one active learning experiment
def run_roberta_active_learning(method_name):
    labeled_df = initial_labeled_df.copy().reset_index(drop=True)
    unlabeled_df = initial_unlabeled_df.copy().reset_index(drop=True)

    results = []

    for round_num in range(N_ROUNDS):
        print(f"\n=== Method: {method_name} | Round {round_num+1}/{N_ROUNDS} ===")
        print("Labeled size:", len(labeled_df))

        trainer, eval_results = train_and_evaluate_roberta(
            labeled_df,
            run_name=f"roberta_{method_name}_round_{round_num}"
        )

        results.append({
            "method": method_name,
            "round": round_num + 1,
            "labeled_size": len(labeled_df),
            "accuracy": eval_results["eval_accuracy"],
            "f1_macro": eval_results["eval_f1_macro"]
        })

        # stop if last round or not enough left
        if round_num == N_ROUNDS - 1 or len(unlabeled_df) < QUERY_SIZE:
            break

        if method_name == "random":
            selected_idx = query_random(
                unlabeled_df,
                query_size=QUERY_SIZE,
                random_state=RANDOM_STATE + round_num
            )

        else:
            probs = get_unlabeled_probs(trainer, unlabeled_df)

            if method_name == "entropy":
                selected_idx = query_entropy(unlabeled_df, probs, QUERY_SIZE)
            elif method_name == "margin":
                selected_idx = query_margin(unlabeled_df, probs, QUERY_SIZE)
            else:
                raise ValueError("Unknown method")

        selected_rows = unlabeled_df.loc[selected_idx].copy()
        labeled_df = pd.concat([labeled_df, selected_rows], ignore_index=True)
        unlabeled_df = unlabeled_df.drop(index=selected_idx).reset_index(drop=True)

    return pd.DataFrame(results)

In [ ]:
#Running all methods
random_results = run_roberta_active_learning("random")
entropy_results = run_roberta_active_learning("entropy")
margin_results = run_roberta_active_learning("margin")

all_results = pd.concat([random_results, entropy_results, margin_results], ignore_index=True)

print("\nAll results:")
print(all_results)

In [ ]:
#Plotting F1 + accuracy across the methods

import matplotlib.pyplot as plt

# Plot macro F1
plt.figure(figsize=(8, 5))

for method in all_results["method"].unique():
    subset = all_results[all_results["method"] == method]
    plt.plot(
        subset["labeled_size"],
        subset["f1_macro"],
        marker="o",
        label=method
    )

plt.xlabel("Number of labeled samples")
plt.ylabel("Macro F1")
plt.title("RoBERTa: Active Learning vs Random Sampling")
plt.legend()
plt.grid(True)
plt.show()

# Plot accuracy
plt.figure(figsize=(8, 5))

for method in all_results["method"].unique():
    subset = all_results[all_results["method"] == method]
    plt.plot(
        subset["labeled_size"],
        subset["accuracy"],
        marker="o",
        label=method
    )

plt.xlabel("Number of labeled samples")
plt.ylabel("Accuracy")
plt.title("RoBERTa Accuracy vs Number of Labeled Samples")
plt.legend()
plt.grid(True)
plt.show()